In [ ]:
# uncomment these lines on a fresh Colab runtime:
# !pip install -q torch transformers nltk rouge-score matplotlib
# !git clone <your-repo-url> cs4782-lora-replication
# %cd cs4782-lora-replication/code

import sys, os, json
sys.path.insert(0, ".")
import matplotlib.pyplot as plt
import torch
print(f"torch={torch.__version__}, cuda available={torch.cuda.is_available()}")

In [ ]:
from transformers import GPT2LMHeadModel
from lora import inject_lora
from sequential_lora import inject_sequential_lora
from sequential_train import run_sequential_experiment, FixedStepTrigger, PlateauTrigger
from data import get_dataloaders, load_e2e_dataset
from train import run_experiment

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
train_loader, val_loader, test_loader, tokenizer = get_dataloaders(batch_size=8, max_length=256)
dataset = load_e2e_dataset()  # raw dicts, needed by generate_texts/compute_metrics
NUM_EPOCHS = 5
TOTAL_STEPS = NUM_EPOCHS * len(train_loader)
print(f"train batches/epoch: {len(train_loader)}, total steps for 5 epochs: {TOTAL_STEPS}")

In [ ]:
# baseline at r=10, alpha=10 (matches project convention alpha=rank)
# this is THE critical comparison: if staged conditions dont beat this,
# staging didn't add anything beyond just training at higher rank.
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.resize_token_embeddings(len(tokenizer))
inject_lora(model, rank=10, alpha=10)

run_experiment(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    test_dataset_hf=dataset["test"],
    tokenizer=tokenizer,
    device=device,
    num_epochs=NUM_EPOCHS,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=500,
    experiment_name="lora_r10",
)

In [ ]:
# 5 stages x rank 2 each = final rank 10. boundaries at 1/5, 2/5, 3/5, 4/5 of training.
for alpha_old, name in [(0.0, "seq_fixed_frozen"),
                        (0.1, "seq_fixed_hybrid"),
                        (1.0, "seq_fixed_unfrozen")]:
    print(f"\n{'=' * 60}\n{name} (alpha_old={alpha_old})\n{'=' * 60}")

    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.resize_token_embeddings(len(tokenizer))
    inject_sequential_lora(model, rank=2, alpha=2)

    trigger = FixedStepTrigger(total_steps=TOTAL_STEPS, num_stages=5)
    run_sequential_experiment(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        test_dataset_hf=dataset["test"],
        tokenizer=tokenizer,
        device=device,
        trigger=trigger,
        per_stage_rank=2,
        per_stage_alpha=2,
        alpha_old=alpha_old,
        num_epochs=NUM_EPOCHS,
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_steps=500,
        eval_every=200,
        experiment_name=name,
    )

In [ ]:
# emergent final rank, capped at 10. patience=3 evals, delta=0.001, eval every 200 steps.
for alpha_old, name in [(0.0, "seq_plateau_frozen"),
                        (0.1, "seq_plateau_hybrid"),
                        (1.0, "seq_plateau_unfrozen")]:
    print(f"\n{'=' * 60}\n{name} (alpha_old={alpha_old})\n{'=' * 60}")

    model = GPT2LMHeadModel.from_pretrained("gpt2")
    model.resize_token_embeddings(len(tokenizer))
    inject_sequential_lora(model, rank=2, alpha=2)

    trigger = PlateauTrigger(
        patience=3, delta=0.001, max_total_rank=10, per_stage_rank=2,
    )
    run_sequential_experiment(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        test_dataset_hf=dataset["test"],
        tokenizer=tokenizer,
        device=device,
        trigger=trigger,
        per_stage_rank=2,
        per_stage_alpha=2,
        alpha_old=alpha_old,
        num_epochs=NUM_EPOCHS,
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_steps=500,
        eval_every=200,
        experiment_name=name,
    )

In [ ]:
def _load(path):
    with open(path) as f:
        return json.load(f)

results = {
    "lora_r2":              _load("../results/metrics/lora_r2.json"),
    "lora_r10":             _load("../results/metrics/lora_r10.json"),
    "seq_fixed_frozen":     _load("../results/metrics/sequential/seq_fixed_frozen.json"),
    "seq_fixed_hybrid":     _load("../results/metrics/sequential/seq_fixed_hybrid.json"),
    "seq_fixed_unfrozen":   _load("../results/metrics/sequential/seq_fixed_unfrozen.json"),
    "seq_plateau_frozen":   _load("../results/metrics/sequential/seq_plateau_frozen.json"),
    "seq_plateau_hybrid":   _load("../results/metrics/sequential/seq_plateau_hybrid.json"),
    "seq_plateau_unfrozen": _load("../results/metrics/sequential/seq_plateau_unfrozen.json"),
}
for k, v in results.items():
    print(f"{k}: BLEU={v['test_metrics']['bleu']:.4f}  ROUGE-L={v['test_metrics']['rouge_l']:.4f}")

In [ ]:
rows = []
for name, r in results.items():
    if "stage_events" in r:
        final_rank = r.get("final_total_rank", "—")
    else:
        final_rank = 2 if "r2" in name else 10
    rows.append({
        "name": name,
        "final_rank": final_rank,
        "trainable": r["params"]["trainable"],
        "bleu": r["test_metrics"]["bleu"],
        "rouge_l": r["test_metrics"]["rouge_l"],
    })

print(f"{'Name':<24} {'Final Rank':>10} {'Trainable':>12} {'BLEU':>8} {'ROUGE-L':>8}")
print("-" * 70)
for row in rows:
    print(f"{row['name']:<24} {str(row['final_rank']):>10} "
          f"{row['trainable']:>12,} {row['bleu']:>8.4f} {row['rouge_l']:>8.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
colors = {
    "lora_r2":              "#888888",
    "lora_r10":             "#000000",
    "seq_fixed_frozen":     "#2196F3",
    "seq_fixed_hybrid":     "#03A9F4",
    "seq_fixed_unfrozen":   "#00BCD4",
    "seq_plateau_frozen":   "#F44336",
    "seq_plateau_hybrid":   "#FF5722",
    "seq_plateau_unfrozen": "#FF9800",
}

for name, r in results.items():
    color = colors[name]
    if "val_loss_log" in r:
        steps = [s for s, _ in r["val_loss_log"]]
        losses = [l for _, l in r["val_loss_log"]]
        ax.plot(steps, losses, label=name, color=color, alpha=0.85)
        for ev in r.get("stage_events", []):
            ax.axvline(ev["step"], color=color, linestyle=":", alpha=0.4)
    else:
        # baselines only have per-epoch val loss; place each at end of its epoch
        steps_per_epoch = len(r["training_history"]["val_loss"])
        # better: use TOTAL_STEPS / num_epochs from cell 3 if you saved it
        steps_per_epoch = 5246  # E2E train: 41961 examples / batch 8
        steps = [(i + 1) * steps_per_epoch for i in range(len(r["training_history"]["val_loss"]))]
        ax.plot(steps, r["training_history"]["val_loss"],
                marker="o", label=name, color=color, linestyle="--", alpha=0.85)

ax.set_xlabel("Training Step")
ax.set_ylabel("Validation Loss")
ax.set_title("Validation Loss Curves: Sequential vs Baselines")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs("../results/figures/sequential", exist_ok=True)
plt.savefig("../results/figures/sequential/val_loss_curves.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names = [row["name"] for row in rows]
bleu = [row["bleu"] for row in rows]
rouge = [row["rouge_l"] for row in rows]
bar_colors = [colors[n] for n in names]

axes[0].bar(names, bleu, color=bar_colors)
axes[0].set_ylabel("BLEU")
axes[0].set_title("BLEU across all 8 conditions")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(names, rouge, color=bar_colors)
axes[1].set_ylabel("ROUGE-L")
axes[1].set_title("ROUGE-L across all 8 conditions")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("../results/figures/sequential/final_metrics_comparison.png", dpi=150)
plt.show()

In [ ]:
# emergent final rank (plateau conditions)
plateau_names = ["seq_plateau_frozen", "seq_plateau_hybrid", "seq_plateau_unfrozen"]
ranks = [results[n]["final_total_rank"] for n in plateau_names]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(plateau_names, ranks, color=[colors[n] for n in plateau_names])
ax.set_ylabel("Emergent Final Rank (cap=10)")
ax.set_title("Plateau Stopping: Emergent Rank by α")
ax.axhline(10, color="black", linestyle="--", alpha=0.3, label="rank cap")
ax.legend()
plt.tight_layout()
plt.savefig("../results/figures/sequential/emergent_ranks.png", dpi=150)
plt.show()

# per-stage val loss (sanity check: did each new stage actually help?)
print(f"\n{'Condition':<24} {'Boundary step':>14} {'Val loss at trans':>18} {'New rank':>10}")
print("-" * 70)
for name in ["seq_fixed_frozen", "seq_fixed_hybrid", "seq_fixed_unfrozen",
             "seq_plateau_frozen", "seq_plateau_hybrid", "seq_plateau_unfrozen"]:
    for ev in results[name].get("stage_events", []):
        vl = ev.get("val_loss_at_transition")
        vl_str = f"{vl:.4f}" if vl is not None else "—"
        print(f"{name:<24} {ev['step']:>14} {vl_str:>18} {ev['new_total_rank']:>10}")
    print()